# 01 · T6 — Datasets and constraint files

**Deliverable (plan, section 3):** `data/processed/<dataset>/*` and `configs/<dataset>.yaml` for the five datasets.

This notebook:
1. downloads the datasets (once) and records **id, version and md5** in `data/manifest.jsonl`;
2. applies the preprocessing decided in the plan (section 3.2);
3. runs the **HELOC missing-value hypothesis tests** (section 3.3);
4. validates the YAML files against the schema and against the processed columns;
5. checks the T6 acceptance checklist (section 3.6).

All the logic lives in `cfqd/datasets.py`; here we only call and inspect it.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

In [2]:
from cfqd import datasets as D
from cfqd.descriptor import Descriptor
from cfqd.schema import Constraint

DATASETS = ["german_credit", "adult", "heloc", "gmc", "compas", "compas_guidotti"]

## 1. Acquisition, preprocessing and manifest

`D.build(name)` downloads only if `data/raw/<base>/SOURCE.json` with a verified md5 is missing, and preprocesses
only if `data/processed/<name>/data.parquet` is missing (`force=True` redoes it).

| Dataset | Source | Note |
|---|---|---|
| German Credit | UCI 522 (South German Credit, Groemping's recoding) | **not** OpenML `credit-g` |
| Adult | UCI 2 (`adult.data` + `adult.test`) | no `fnlwgt`, `education-num` only |
| HELOC | OpenML 45554 v2 (md5 pinned in the YAML) | cite FICO |
| Give Me Some Credit | OpenML 45577 (same data as Kaggle) | |
| COMPAS | ProPublica (GitHub) | standard filter → 6,172 rows |

In [3]:
rows = []
for n in DATASETS:
    df, meta = D.build(n)
    cfg = D.load_config(n)
    rows.append(dict(dataset=n, rows=meta["n_rows"], features=meta["n_features"],
                     favorable_rate=meta["favorable_rate"], favorable_label=cfg["favorable_class"],
                     groups=len(cfg["groups"])))
pd.DataFrame(rows)

,dataset,rows,features,favorable_rate,favorable_label,groups
0,german_credit,1000,20,0.7000,1,4
1,adult,45222,12,0.2478,>50K,4
2,heloc,9871,26,0.4797,Good,4
3,gmc,120121,10,0.9311,0,3
4,compas,6172,9,0.5449,0,0
5,compas_guidotti,6172,9,0.5449,0,1


In [4]:
pd.read_json(D.DATA_DIR / "manifest.jsonl", lines=True)

,dataset,kind,id,version,md5,url,n_rows,n_features,favorable_rate,built_at,generator_hash
0,german_credit,uci,522.0,NaN,f2ab24d5398cc5032615ac6e7a875d4b,https://archive.ics.uci.edu/static/public/522/south+german+credit.zip,1000,20,0.7000,2026-11-20 17:05:19+00:00,None
1,adult,uci,2.0,NaN,1252efa3d4fd9b7a70b3460adb3deb28,https://archive.ics.uci.edu/static/public/2/adult.zip,45222,12,0.2478,2026-11-20 17:05:23+00:00,None
2,heloc,openml,45554.0,2.0,2c675bd4d9cfc63d97771af7a65fd417,https://openml.org/data/v1/download/22116522/FICO-HELOC-cleaned.arff,9871,26,0.4797,2026-11-20 17:05:49+00:00,None
3,gmc,openml,45577.0,1.0,53fbc48b682ecfac16697dfea4cb3c39,https://openml.org/data/v1/download/22116561/Give-Me-Some-Credit.arff,120121,10,0.9311,2026-11-20 17:07:14+00:00,None
4,compas,url,NaN,NaN,9165d40c400bba93a8cffece2b74622b,https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-s...,6172,9,0.5449,2026-11-20 17:07:21+00:00,None
5,compas_guidotti,url,NaN,NaN,9165d40c400bba93a8cffece2b74622b,https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-s...,6172,9,0.5449,2026-11-20 17:07:21+00:00,None
6,simulation,synthetic,NaN,NaN,None,None,6000,11,0.1872,2026-11-20 17:21:57+00:00,82ede74c5d75
7,simulation_shuffled,synthetic,NaN,NaN,None,None,6000,11,0.1872,2026-11-20 17:21:58+00:00,b9ec5696cc86


Preprocessing notes (full details in `data/processed/<dataset>/meta.json`):

In [5]:
import json
for n in ["adult", "gmc", "compas"]:
    meta = json.loads((D.processed_dir(n) / "meta.json").read_text(encoding="utf-8"))
    keep = {k: v for k, v in meta.items() if k.startswith(("rows", "removed", "winsor", "length"))}
    print(n, json.dumps(keep, indent=1))

adult {
 "rows_dropped_missing": 3620,
 "removed_columns": [
  "fnlwgt",
  "education"
 ]
}
gmc {
 "rows_dropped_missing": 29731,
 "rows_dropped_error_codes": 148,
 "winsorize_upper": {
  "RevolvingUtilizationOfUnsecuredLines": {
   "cap": 1.323631781199995,
   "n_capped": 601
  },
  "DebtRatio": {
   "cap": 1782.3999999999942,
   "n_capped": 601
  },
  "MonthlyIncome": {
   "cap": 35000.0,
   "n_capped": 589
  }
 }
}
compas {
 "rows_raw": 7214,
 "rows_filtered": 6172,
 "length_of_stay_missing": 0
}


## 2. HELOC — testing "missing = never happened" (plan 3.3)

Each column with many missing values is crossed with a sibling column that has none:

* **necessity** = P(sibling == expected | column missing) — the hypothesis itself;
* **sufficiency** = P(column missing | sibling == expected) — informative only.

Criterion: the hypothesis is confirmed if necessity ≥ 0.95.

In [6]:
heloc_raw = pd.read_parquet(D.raw_dir(D.load_config("heloc")) / "frame.parquet")
print("rows:", len(heloc_raw), "| with any missing:", int(heloc_raw.isna().any(axis=1).sum()),
      "| left if dropped:", int((~heloc_raw.isna().any(axis=1)).sum()))
tests = D.heloc_missing_tests(heloc_raw)
tests

rows: 9871 | with any missing: 7369 | left if dropped: 2502


,column,missing,missing_pct,sibling,expected,necessity,sufficiency,n_missing_but_sibling_differs,confirmed
0,MSinceMostRecentDelq,4840,49.0,PercentTradesNeverDelq,100,0.951860,1.000000,233,True
1,NetFractionInstallBurden,3419,34.6,PercentInstallTrades,0,0.056449,1.000000,3226,False
2,MSinceMostRecentInqexcl7days,2331,23.6,NumInqLast6Mexcl7days,0,0.539254,0.308239,1074,False


### Reading

* **`MSinceMostRecentDelq` — confirmed** (≈95%): missing = "never delinquent". Median imputation would turn
  "never delinquent" into "delinquent N months ago" for half of the dataset, always for the worse.
* **`NetFractionInstallBurden` — rejected**: almost every missing row **does** have installment trades
  (`PercentInstallTrades` > 0). Here missing does not mean "does not apply".
* **`MSinceMostRecentInqexcl7days` — rejected**: about half of the missing rows have an inquiry in the last 6 months.

**Treatment adopted (pending a group decision):** indicator `<col>_NA` (excluded from mutation) + sentinel `-1`
on the three columns. For the two rejected columns the sentinel now means **"unknown"**: freezing an unknown value is
still the conservative choice (a counterfactual that "changes" a value we do not know is not recourse) and avoids the
artificial median spike that LOF / nearest-neighbor plausibility would read as a dense region. This must be
**stated in the paper**: the meaning of `-9/-8/-7` was *inferred*, and only one of the three inferences held.

The sentinel rule is per instance (`Feature.sentinel`, enforced in `operators.repair` and counted by
`metrics.constraint_violations`): if the original instance holds the sentinel the feature is frozen; otherwise it can
never move into it.

In [7]:
# Extra hint: NumInstallTradesWBalance has an observed minimum of 1 and ALL its missing rows
# are also missing in NetFractionInstallBurden -> missing probably means 0.
a = heloc_raw["NumInstallTradesWBalance"]
print("observed minimum:", a.min(), "| missing:", int(a.isna().sum()))
pd.crosstab(heloc_raw["NetFractionInstallBurden"].isna().rename("NetFractionInstallBurden missing"),
            a.isna().rename("NumInstallTradesWBalance missing"))

observed minimum: 1.0 | missing: 861


NumInstallTradesWBalance missing,False,True
NetFractionInstallBurden missing,,
False,6452,0
True,2558,861


## 3. Validating the YAML files (acceptance criteria)

`D.validate_config` checks: required keys; valid types and constraint categories; **no feature listed twice**
(i.e. in two categories); **every feature is in a group, or frozen, or declared in `ungrouped`**; every group has at
least one mutable feature; and, given the data, that the YAML features are exactly the processed columns.

In [8]:
validation = []
for n in D.list_dataset_configs():
    df, _ = D.build(n)
    errs = D.validate_config(D.load_config(n), columns=list(df.columns))
    validation.append(dict(config=n, ok=not errs, errors="; ".join(errs)))
pd.DataFrame(validation)

,config,ok,errors
0,adult,True,
1,compas,True,
2,compas_guidotti,True,
3,german_credit,True,
4,gmc,True,
5,heloc,True,
6,simulation,True,
7,simulation_shuffled,True,


## 4. Constraints and groups per dataset (check against table 3.5 of the plan)

In [9]:
def constraint_summary(n):
    cfg = D.load_config(n)
    by = {}
    for f in cfg["features"]:
        by.setdefault(f.get("constraint", "free"), []).append(f["name"])
    return dict(dataset=n, g=len(cfg["groups"]),
                immutable=", ".join(by.get("immutable", [])),
                increase_only=", ".join(by.get("increase_only", [])),
                irreversible=", ".join(by.get("irreversible", [])),
                excluded=", ".join(by.get("excluded", [])),
                ungrouped=", ".join(cfg.get("ungrouped") or []))

pd.DataFrame([constraint_summary(n) for n in DATASETS]).set_index("dataset")

,g,immutable,increase_only,irreversible,excluded,ungrouped
dataset,,,,,,
german_credit,4,"personal_status_sex, age, people_liable, foreign_worker","employment_duration, present_residence",credit_history,,"present_residence, telephone"
adult,4,"marital-status, relationship, race, sex, native-country","age, education-num",,,age
heloc,4,,"MSinceOldestTradeOpen, AverageMInFile, MSinceMostRecentDelq, MSinceMostRecen...","NumTrades60Ever2DerogPubRec, NumTrades90Ever2DerogPubRec, PercentTradesNever...","ExternalRiskEstimate, MSinceMostRecentDelq_NA, MSinceMostRecentInqexcl7days_...","MSinceOldestTradeOpen, MSinceMostRecentTradeOpen, AverageMInFile, MSinceMost..."
gmc,3,,age,"NumberOfTime30-59DaysPastDueNotWorse, NumberOfTimes90DaysLate, NumberOfTime6...",,age
compas,0,"age, sex, race, c_charge_degree",,"priors_count, juv_fel_count, juv_misd_count, juv_other_count, length_of_stay",,
compas_guidotti,1,"age, sex, race, c_charge_degree",,,,


In [10]:
ds = D.load_dataset("german_credit")
D.describe_spec(ds.spec)

,feature,type,constraint,group,low,high,n_categories,mad,sentinel
0,status,ordinal,free,accounts,0.0,3.0,4.0,1.000000,None
1,duration,numeric,free,contract,4.0,72.0,NaN,6.000000,None
2,credit_history,ordinal,irreversible,None,0.0,4.0,5.0,1.082578,None
3,purpose,nominal,free,contract,0.0,10.0,11.0,1.000000,None
4,amount,numeric,free,contract,250.0,18424.0,NaN,1097.500000,None
5,savings,ordinal,free,accounts,0.0,4.0,5.0,1.579232,None
6,employment_duration,ordinal,increase_only,employment_capacity,0.0,4.0,5.0,1.000000,None
7,installment_rate,ordinal,free,contract,0.0,3.0,4.0,1.000000,None
8,personal_status_sex,nominal,immutable,None,0.0,3.0,4.0,0.707726,None
9,other_debtors,nominal,free,collateral,0.0,2.0,3.0,0.477467,None


## 5. Size of the strategy map per dataset

Coverage denominator: **feasible** cells (strategies reachable under the constraints × effort bins). This number
feeds pending item 4 of the plan (single budget vs. curves): with 4 groups the grid exceeds 400 cells.
Strict COMPAS has **0** groups: the map is empty and no method can run.

In [11]:
cells = []
for n in DATASETS + ["simulation"]:
    spec = D.load_dataset(n).spec
    d = Descriptor(spec)
    cells.append(dict(dataset=n, g=d.g, feasible_strategies=len(d.feasible_strategies()),
                      effort_bins=d.n_effort, feasible_cells=d.n_feasible_cells,
                      total_cells=d.n_total_cells, mutable_features=int(spec.mutable_mask.sum())))
pd.DataFrame(cells)

,dataset,g,feasible_strategies,effort_bins,feasible_cells,total_cells,mutable_features
0,german_credit,4,80,5,400,400,15
1,adult,4,35,5,175,400,7
2,heloc,4,80,5,400,400,17
3,gmc,3,26,5,130,130,7
4,compas,0,0,5,0,0,0
5,compas_guidotti,1,2,5,10,10,5
6,simulation,4,80,5,400,400,10


## 6. T6 acceptance checklist (plan 3.6)

In [12]:
manifest = pd.read_json(D.DATA_DIR / "manifest.jsonl", lines=True).set_index("dataset")
checks = {
    "all YAML files pass the schema validator": all(r["ok"] for r in validation),
    "no feature in two constraint categories (also tested in tests/test_datasets.py)": all(r["ok"] for r in validation),
    "every feature typed and in a group, or declared out": all(r["ok"] for r in validation),
    "the three HELOC hypothesis tests ran and are stored in meta.json":
        len(json.loads((D.processed_dir("heloc") / "meta.json").read_text())["missing_tests"]) == 3,
    "the manifest has source and md5 for every dataset":
        all(pd.notna(manifest.loc[b, "md5"]) for b in ["german_credit", "adult", "heloc", "gmc", "compas"]),
}
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)

PASS all YAML files pass the schema validator
PASS no feature in two constraint categories (also tested in tests/test_datasets.py)
PASS every feature typed and in a group, or declared out
PASS the three HELOC hypothesis tests ran and are stored in meta.json
PASS the manifest has source and md5 for every dataset
